# Generating Displaced Coordinates from Survey Data
This notebook serves two purposese as follows:
1. **Usage example.** A walkthough of how to use the ```point_displacement``` module to generate 
displaced coordinates from household survey GPS coordinates data.
2. **Benchmark results from ```point_displacement```.** Use results from the ArcGIS version of the process to test whether the results from ```point_displacement``` are correct.

## Python setup

In [11]:
from pathlib import Path
import pandas as pd

from spatial_anonymization import point_displacement 
from spatial_anonymization import utils 

## Usage example
This example demonstrates how to use the ```point_displacement``` module to generate 
displaced coordinates which can then be shared with the public.

 ### Step-1. Input/output data setup
Provide absolute paths to the folliwing files:
1. **GPS coordinates file.** This is the file with GPS coordinates which need to be displaced. For now, we are supporting only **CSV** and **STATA** ```.dta``` file formats.
2. **Administrative regions.**  A shapefile (or other geographic formats such as Geojson) of admin regions within which the displacements will be confined.
3. **outputs directory.** Provide directory where outputs from running the processing will be saved.

In [12]:
# base data folder
DATA = Path.cwd().parents[0].joinpath('data')

# ========================
# INPUT FILES FOR ETHIOPIA
# ========================
ETH_SURVEY = DATA.joinpath('ETH', 'survey')
ETH_GEOVARS = DATA.joinpath('ETH', 'w4_geovars')

# GPS coordinates file
# ETH_COORDS = ETH_SURVEY.joinpath("HH level coordinates.dta")
ETH_COORDS = DATA.joinpath("ETH/survey/ETH_gps_work_r12082021.csv")

# Admin regions (adm2) shapefile
ETH_ADM = ETH_GEOVARS.joinpath("spatial_data", "shp_adm2", "eth_admbnda_adm2_csa_bofed_20190827.shp")

# output directory
ETH_OUPUT_DIR = DATA.joinpath("ETH", "outputs")

### Step-2. Parameters and other global variables setup
Now, we setep parameters as well as helper global variables as follows:
#### Displacement parameters
All distance paramters are in meters
1. **urb_max_dist.** The maximum distance to displace a point in urban areas.
2. **rur_max_dist.** The maximum distance to displace a point in rural areas.
3. **rur_max_dist_subset.** The maximum distance to displace a point in rural areas for a subset of the households. 
4. **urb_min_dist.** Allowed minimum distance between original point and displaced point in urban areas.
5. **rur_min_dist.** Allowed minimum distance between original point and displaced point in rural areas.
6. **apply_min_dist.** whether to apply minimum distance threshold or not
7. **rur_subset_prop.** proportion of rural households to apply special maximum distance

#### Processing variables/args 
1. **geodetic_crs_epsg.** The geographic CRS for which the latitude and longitude were collected. The default is WGS84 given as EPSG code. If you have the datum-name or CRS details but do not know the EPSG code, please use [epsg.io](https://epsg.io)
2. **lon_col.** Column name containing longitude
3. **lat_col.** Column name containing latitude
4. **uniq_id.** Column name (if available uniquely identifying each row
5. **urb_rur_col.** Column name indicating whether point is in urban or rural
6. **max_tries.** In some cases where there are many coordinates and the confining polygon is small, finding displaced coordinates withing the polygon takes too long and we give up after these many tries.

In [13]:
# =====================
# SETUP PARAMS 
# ====================
# urb_max_dist
urb_max = 2000
# rur_max_dist
rur_max = 5000
# rur_max_dist_subset
rur_max_subset = 10000
# urb_min_dist
urb_min = 100
# rur_min_dist
rur_min = 1000
# apply_min_dist
apply_min_dist = False
# rur_subset_prop = 10%
rur_sub_prop = 0.1
# max_tries
max_tries = 30

# setup the params dict
PARAMS = {'urb_max_dist': urb_max, 'rur_max_dist': rur_max, 'rur_max_dist_subset':rur_max_subset,
          'urb_min_dist': urb_min, 'rur_min_dist': rur_min, 'maximum_tries': max_tries,
          'apply_min_dist': apply_min_dist, 'rur_sub_prop': rur_sub_prop}



# ========================
# PREPARE PROCESSING ARGS
# =======================
# add EPSG code for geographic CRS if its not 
# default to WGS84
GEO_CRS = 4326
LON ='saq19__Longitude'
LAT ='saq19__Latitude'
EA_ID = 'ea_id'
URB_RUR = 'saq14'
RURAL_CODE = 'RURAL'

### Step-3. Run the displacement

In [14]:
# =========================
# CREATE DISPLACER OBJECT
# =========================
eth_displacer = point_displacement.SimpleDisplacer(PARAMS, input_coords=ETH_COORDS,  
                                               shp_adm_bounds=ETH_ADM, output_dir=ETH_OUPUT_DIR,
                                               lon_col=LON, lat_col=LAT, urb_rur_col=URB_RUR, 
                                                   unique_id=EA_ID, rural_code=RURAL_CODE, 
                                                   geodetic_crs=GEO_CRS)


# =========================
# RUN DISPLACEMENT PROCESS
# =========================
# Provide full path to output CSV file. If not provided, otherwise
# Output will be saved to 'output_dir' used above with ETH_COORDS + suffix disp
out_fpath = ETH_OUPUT_DIR.joinpath("w4_ea_level_displacements.csv")
eth_displacer.run_displacer(output_file=out_fpath)

PREPARING DATA FOR DISPLACEMENT...........
################################################## 


135 null lat/lon rows dropped out of 6,894

Dropped observations saved to file: 
 /Users/dunstanmatekenya/Library/CloudStorage/OneDrive-WBG/Spatial-Anonymization/data/ETH/outputs/ETH_gps_work_r12082021_dropped.csv

Grouping to ea_id level and calculating average coordinates

Found the following projected CRS: ['32637', '32636', '32638']
ADDING DISPLACED POINTS...........
################################################## 

RUNNING SOME CHECKS .........
################################################## 

SAVING OUTPUTS .........
################################################## 


Output CSV file with anonymized coordinates saved to file: 
 /Users/dunstanmatekenya/Library/CloudStorage/OneDrive-WBG/Spatial-Anonymization/data/ETH/outputs/w4_ea_level_displacements.csv

STATISTICS BASED OFF ORIGINAL VS. DISPLACED COORDS
      stat   area     value
0     mean  urban  0.941512
1     mean  rural 

## Benchmark Python results with those from ArcGIS
### Compare displacement statistics
For this, we will compare summary statistics based on distance between the original coordinates and displaced coordinates.

### Compare geovariables
The ultimate goal of generating anonymized GPS coordinates is so that the public can use them to generate spatial variables. 

In this, we comapre geovariables values generated from displaced coordinates from the Python module against those from ArcGIS.

### Comparing displacement statistics

In [15]:
# File containing displaced coordinates from Python module
ETH_coords_disp_py = DATA.joinpath("ETH/outputs/ETH_gps_work_r12082021_disp.csv")

# File containing displaced coordinates from ArcGIS
ETH_coords_disp_arc = DATA.joinpath("ETH/survey/ETH_gps_work_r12082021.csv")

In [16]:
# Load the data
df_py = pd.read_csv(ETH_coords_disp_py)
df_arc = pd.read_csv(ETH_coords_disp_arc)

# Lets make sure we have the same EAs
# Although it doesnt matter for summaries
df_arc.drop_duplicates(subset=['ea_id'], inplace=True)
df_arc.dropna(subset=[LON, LAT], inplace=True)

# Merge the two files
df = df_arc.merge(df_py, on=EA_ID, how='left', indicator=True)

assert df._merge.value_counts()['both'] == len(df_arc)

In [17]:
print('Displaying dispacement statistics based on Python module ....')
df_py_stats = point_displacement.generate_displacement_stats(df_py, x='ea_ctr_lon', y='ea_ctr_lat',
                                                             x_disp='disp_lon', y_disp='disp_lat', 
                                               urb_rur_col='urb_rur', rural_val=1,
                                               stats=['mean', 'median', 'max', 'min'])

Displaying dispacement statistics based on Python module ....

STATISTICS BASED OFF ORIGINAL VS. DISPLACED COORDS
      stat   area     value
0     mean  urban  1.009733
1     mean  rural  2.867315
2   median  urban  1.048705
3   median  rural  2.728417
4  minimum  urban  0.007341
5  minimum  rural  0.009282
6  maximum  urban  1.987138
7  maximum  rural  9.719898


In [18]:
print('Displaying dispacement statistics based on ArcGIS module ....')
df_arc_stats = point_displacement.generate_displacement_stats(df_arc, x='ea_ctr_lon', y='ea_ctr_lat',
                                                             x_disp='lon_dd_mod', y_disp='lat_dd_mod', 
                                               urb_rur_col='saq14', rural_val='RURAL',
                                               stats=['mean', 'median', 'max', 'min'])

Displaying dispacement statistics based on ArcGIS module ....

STATISTICS BASED OFF ORIGINAL VS. DISPLACED COORDS
      stat   area     value
0     mean  urban  0.974498
1     mean  rural  2.679262
2   median  urban  0.967353
3   median  rural  2.529610
4  minimum  urban  0.001130
5  minimum  rural  0.005057
6  maximum  urban  1.979333
7  maximum  rural  8.657019


**The summary statistics look very similar.**